In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / 'pipeline_config.yaml').exists() and (root.parent / 'pipeline_config.yaml').exists():
    root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Working directory: {root}')


Working directory: c:\Users\Usuario\OneDrive\Escritorio\preProcessing


# Phase 5 - Integration and Validation

This notebook verifies the integrated feature matrix artifacts produced by the pipeline and documents final feature counts.

In [2]:
from pathlib import Path
import pandas as pd

from run_pipeline import load_config

In [3]:
config = load_config()
paths = config.get('paths', {})
feature_matrix_path = Path(paths.get('feature_matrix', 'datasets/features/feature_matrix.parquet'))
pipeline_model_path = Path(paths.get('pipeline_model', 'models/preprocessing_pipeline.joblib'))
validation_report_path = Path('reports/data_validation_report.html')

assert feature_matrix_path.exists(), f'Missing feature matrix: {feature_matrix_path}'
assert pipeline_model_path.exists(), f'Missing serialized pipeline: {pipeline_model_path}'
assert validation_report_path.exists(), f'Missing validation report: {validation_report_path}'

feature_df = pd.read_parquet(feature_matrix_path)
feature_df['date'] = pd.to_datetime(feature_df['date'], utc=True, errors='coerce')
feature_df['target_start_date'] = pd.to_datetime(feature_df['target_start_date'], utc=True, errors='coerce')
feature_df = feature_df.sort_values(['ticker', 'date']).reset_index(drop=True)

print('Loaded:', feature_matrix_path)
print('Rows:', len(feature_df), '| Columns:', len(feature_df.columns))

Loaded: datasets\features\feature_matrix.parquet
Rows: 4178400 | Columns: 38


In [4]:
identifier_cols = {'ticker', 'date', 'dataset_split', 'target_realized_vol_5d', 'target_start_date'}
feature_cols = [c for c in feature_df.columns if c not in identifier_cols]

summary = {
    'rows': int(len(feature_df)),
    'columns': int(len(feature_df.columns)),
    'feature_count': int(len(feature_cols)),
    'train_rows': int((feature_df['dataset_split'] == 'train').sum()) if 'dataset_split' in feature_df.columns else None,
    'validation_rows': int((feature_df['dataset_split'] == 'validation').sum()) if 'dataset_split' in feature_df.columns else None,
    'test_rows': int((feature_df['dataset_split'] == 'test').sum()) if 'dataset_split' in feature_df.columns else None,
    'rows_without_news_pct': float((feature_df.get('news_count', pd.Series(index=feature_df.index)).fillna(0) == 0).mean())
}

pd.Series(summary)

rows                     4.178400e+06
columns                  3.800000e+01
feature_count            3.300000e+01
train_rows               2.811067e+06
validation_rows          7.886670e+05
test_rows                5.786660e+05
rows_without_news_pct    9.647008e-01
dtype: float64

In [5]:
assert feature_df['ticker'].notna().all()
assert feature_df['date'].notna().all()
assert feature_df['target_realized_vol_5d'].notna().all()
assert ((feature_df['realized_vol_5d'].dropna() > 0).all())
assert ((feature_df['target_start_date'].dropna() > feature_df.loc[feature_df['target_start_date'].notna(), 'date']).all())
assert feature_df.groupby('ticker', observed=True)['date'].apply(lambda s: bool(s.is_monotonic_increasing)).all()

print('Integration sanity checks: PASS')

Integration sanity checks: PASS


In [8]:
validation_tables = pd.read_html(validation_report_path)
print('Validation report tables found:', len(validation_tables))
validation_tables[2]

Validation report tables found: 4


,label,success,evaluated_expectations,successful_expectations,unsuccessful_expectations
0,train_target_not_null,True,1,1,0
1,realized_vol_5d_positive,True,1,1,0
2,target_window_after_feature_date,True,1,1,0
